In [1]:
import numpy as np
import random

In [3]:
#Define State and Actions
states = np.arange(0, 101, 10) # 0, 10,20, 30,40 .......... 100
actions = ['FILL', 'STOP' ]    #Fill or Stop
print('States:', states)
print('Actions:', actions)

States: [  0  10  20  30  40  50  60  70  80  90 100]
Actions: ['FILL', 'STOP']


In [4]:
#create Q table and set alpha, gamma, epsilon, episodes
#Q Table: [s,a]
Q = np.zeros((len(states), len(actions)))

alpha = 0.1 # Learning rate
gamma = 0.9 # Discount factor
epsilon = 0.2 # Exploration rate
episodes = 300 # Training runs

print('Q-table shape:', Q.shape)

Q-table shape: (11, 2)


In [6]:
def get_reward(level, action):
    if 40 <= level <= 70:
        reward = 10  # ideal water level range
    else:
        reward = -10  # too low or too high
    
    # Penalty if filling when the tank is almost full
    if action == 'FILL' and level >= 90:
        reward -= 10  # risk of overflow
    
    # Penalty if stopping when the tank is almost empty
    if action == 'STOP' and level <= 10:
        reward -= 10  # risk of running empty
    
    return reward
print ('Reward Example (level=60, FILL):',get_reward(60, 'FILL'))

Reward Example (level=60, FILL): 10


In [7]:
def next_level(level, action):
    if action == 'FILL':
        level += random.choice([5, 10, 15])
    else:
        level -= random.choice([5, 10, 15])
    return int(np.clip(level, 0, 100))

print('Next Level Example:', next_level(50, 'FILL'))

Next Level Example: 65


In [13]:
# Q-learning training loop for water tank control
for ep in range(episodes):
    # Start with a random water level
    level = random.choice(states)

    for _ in range(15):  # steps per episode
        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon:
            # Explore: choose a random action
            action = random.choice(actions)
        else:
            # Exploit: choose best action from Q-table
            action = actions[np.argmax(Q[level // 10])]

        # Environment updates the water level based on the action
        next_state = next_level(level, action)

        # Get reward for taking this action at the new level
        reward = get_reward(next_state, action)

        # --- Q-learning update ---
        a = actions.index(action)  # action index
        best_next = np.max(Q[next_state // 10])  # best future reward
        Q[level // 10, a] += alpha * (reward + gamma * best_next - Q[level // 10, a])

        # Move to the next state (level changes)
        level = next_state
        
print('Training Completed')

Training Completed


In [15]:
# Ask user for starting water level
try:
#    level = int(input('Enter starting water level (0-100): '))
    if level < 0 or level > 100:
        raise ValueError('Water level out of range!')
except ValueError as e:
    print(e)
    level = 50  # default level if input invalid
    print('Default level set to 50%.')

print(f'\nStarting with level: {level}%')
print('Simulating for 10 steps:\n')

# Simulate 10 steps of Q-learning policy
for step in range(10):
    # Choose the best action based on the learned Q-table
    action = actions[np.argmax(Q[level // 10])]
    
    # Print current step, water level, and chosen action
    print(f'Step {step+1}: Level={level}% -> Action={action}')
    
    # Update the water level based on the action
    level = next_level(level, action)

print('\nSimulation complete. Water tank control finished.')


Enter starting water level (0-100):  0



Starting with level: 0%
Simulating for 10 steps:

Step 1: Level=0% -> Action=FILL
Step 2: Level=15% -> Action=FILL
Step 3: Level=30% -> Action=FILL
Step 4: Level=35% -> Action=FILL
Step 5: Level=50% -> Action=FILL
Step 6: Level=60% -> Action=STOP
Step 7: Level=55% -> Action=FILL
Step 8: Level=60% -> Action=STOP
Step 9: Level=55% -> Action=FILL
Step 10: Level=65% -> Action=STOP

Simulation complete. Water tank control finished.
